# Segment Every Grain — ResNeXt variant

Same workflow as `Segment_every_grain.ipynb`, but the first-pass semantic segmentation is produced by the PyTorch **MaskingResNeXt** model (Satterlee et al., 2025) instead of the Keras UNet. SAM still does the second-pass instance segmentation.

The ResNeXt outputs the same `(H, W, 3)` softmax probabilities (background / interior / boundary) as the UNet, so the rest of the pipeline (`label_grains`, `sam_segmentation`, the interactive editor, etc.) is unchanged.

The fine-tuning section at the bottom mirrors `Train_ResNeXt_model.ipynb`, but loads the existing checkpoint as a starting point so you can adapt the model to a new image domain without retraining from scratch.

The large-image (`predict_large_image`) workflow is intentionally omitted — see the original notebook for that.

## Import packages

In [11]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

import segment_anything
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from segmenteverygrain.resnext_model import (
    MaskingResNeXt,
    load_resnext,
    weighted_crossentropy_torch,
)

%matplotlib qt

device = (
    'mps' if torch.backends.mps.is_available()
    else ('cuda' if torch.cuda.is_available() else 'cpu')
)
print(f'Using device: {device}')

Using device: mps


## Load models

In [14]:
# Load ResNeXt (PyTorch) — replaces the Keras UNet for first-pass inference.
resnext_path = './models/resnext_model_finetuned.pth'
resnext = load_resnext(resnext_path, device=device).to(device)
resnext.eval()

# Download SAM model (only downloads it if it does not exist)
if not os.path.exists('./models/sam_vit_h_4b8939.pth'):
    import urllib.request
    url = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
    urllib.request.urlretrieve(url, './models/sam_vit_h_4b8939.pth')

sam = segment_anything.sam_model_registry['default'](
    checkpoint='./models/sam_vit_h_4b8939.pth'
)
predictor = segment_anything.SamPredictor(sam)

### Keras-style adapter

`seg.predict_image` performs the windowed-tile blending and calls `model.predict(batch, verbose=0)` on each tile — a Keras convention. The wrapper below presents the PyTorch ResNeXt with the same interface, so the full UNet inference pipeline works unchanged.

In [15]:
class TorchKerasAdapter:
    """Adapt a torch segmentation model to the Keras `.predict` interface.

    Accepts a NHWC float batch in [0, 1], returns a NHWC float batch.
    """

    def __init__(self, model, device):
        self.model = model
        self.device = device

    def predict(self, batch, verbose=0):
        x = torch.from_numpy(np.ascontiguousarray(batch).astype('float32'))
        x = x.permute(0, 3, 1, 2).to(self.device)
        with torch.no_grad():
            out = self.model(x)
        return out.permute(0, 2, 3, 1).cpu().numpy()


resnext_adapter = TorchKerasAdapter(resnext, device)

## Run segmentation

Grains are supposed to be well defined in the image; e.g., if a grain consists of only a few pixels, it is unlikely to be detected.

Images with ~2000 pixels along their largest dimension are a good start and allow the user to get an idea about how well the segmentation works.

In [22]:
# replace this with the path to your image:
fname = './Images/cropped_alumina_highquality/cropped_prac7_etched_096.tif'
fname = './Images/20251009-Imperfect-images-&-Doped-sample/Doped-sample/Wilson180_0.25__surface1_01.tif'

image = si.load_image(fname)
predictor.set_image(image)

# First-pass semantic segmentation with the ResNeXt (drop-in for UNet).
image_pred = seg.predict_image(image, resnext_adapter, I=256)

# decreasing the 'dbs_max_dist' parameter results in more SAM prompts
# (and longer processing times):
labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=60.0)

segmenting image tiles...


100%|██████████| 5/5 [00:03<00:00,  1.60it/s]


Check the quality of the ResNeXt labeling and the distribution of SAM prompts (= black dots). If the prediction is poor, head down to the **Fine-tuning** section to adapt the model to your image domain.

In [23]:
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(image_pred)
plt.scatter(np.array(coords)[:, 0], np.array(coords)[:, 1], c='k')
plt.xticks([])
plt.yticks([]);

### Confidence overlays (optional)

In [18]:
plt.figure(figsize=(8, 8))
plt.imshow(image, cmap='gray')
plt.imshow(image_pred[:, :, 0], cmap='magma', alpha=0.5)
plt.colorbar(label='Background probability')
plt.title('ResNeXt background confidence overlay')
plt.axis('off')

(np.float64(-0.5), np.float64(1023.5), np.float64(702.5), np.float64(-0.5))

In [19]:
plt.figure(figsize=(8, 8))
plt.imshow(image, cmap='gray')
plt.imshow(image_pred[:, :, 1], cmap='magma', alpha=0.5)
plt.colorbar(label='Interior probability')
plt.title('ResNeXt interior confidence overlay')
plt.axis('off')

(np.float64(-0.5), np.float64(1023.5), np.float64(702.5), np.float64(-0.5))

In [20]:
plt.figure(figsize=(8, 8))
plt.imshow(image, cmap='gray')
plt.imshow(image_pred[:, :, 2], cmap='magma', alpha=0.5)
plt.colorbar(label='Boundary probability')
plt.title('ResNeXt boundary confidence overlay')
plt.axis('off')

(np.float64(-0.5), np.float64(1023.5), np.float64(702.5), np.float64(-0.5))

In [21]:
# SAM segmentation, using the point prompts from the ResNeXt:
all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
    sam, image, image_pred, coords, labels,
    min_area=300.0,
    plot_image=True,
    remove_edge_grains=True,
    remove_large_objects=False,
)

creating masks using SAM...


100%|██████████| 6023/6023 [06:45<00:00, 14.86it/s]  


finding overlapping polygons...


2676it [06:57,  6.40it/s]


finding best polygons...


100%|██████████| 56/56 [03:38<00:00,  3.89s/it]


creating labeled image...


100%|██████████| 94/94 [00:00<00:00, 438.29it/s]


In [ ]:
out_fn = './prediction_outputs/cropped_prac7_etched_096'
fig.savefig(out_fn + '_grains.jpg', bbox_inches='tight', pad_inches=0)
plt.close()

In [ ]:
# Save first-pass + SAM outputs as training masks (0=background, 1=interior, 2=boundary).
# These can be fed directly into the fine-tuning section below.
fname_stem = os.path.splitext(os.path.basename(fname))[0]
seg.save_training_masks(
    image,
    image_pred,
    mask_all,
    out_stem=os.path.join('./prediction_outputs', fname_stem),
)

## Results

In [ ]:
grains = si.polygons_to_grains(all_grains, image=image)
for g in tqdm(grains, desc='Measuring detected grains'):
    g.measure()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
plt.xticks([])
plt.yticks([])
seg.plot_image_w_colorful_grains(image, all_grains, ax, cmap='Paired', plot_image=True)
seg.plot_grain_axes_and_centroids(all_grains, labels, ax, linewidth=1, markersize=10)
plt.xlim([0, np.shape(image)[1]])
plt.ylim([np.shape(image)[0], 0]);

Run this cell and then click (left mouse button) on one end of the scale bar in the image and click (right mouse button) on the other end of the scale bar:

In [ ]:
cid = fig.canvas.mpl_connect(
    'button_press_event', lambda event: seg.click_for_scale(event, ax)
)

If `px_per_m` is 1, the summary data and histogram will be in pixels. If the ratio of pixels to meters is known, set `px_per_m` to save them in meters.

In [ ]:
# Save results
px_per_m = 3711.9    # 371.19 pixels / 10 cm (scale bar on photo)
out_fn = './prediction_outputs/cropped_prac7_etched_096'
# Grain shapes
si.save_grains(out_fn + '_grains.geojson', grains)
# Summary data
summary = si.save_summary(out_fn + '_summary.csv', grains, px_per_m=px_per_m)
# Summary histogram
si.save_histogram(out_fn + '_summary.jpg', summary=summary)
# Training mask
si.save_mask(out_fn + '_mask.png', grains, image, scale=False)
si.save_mask(out_fn + '_mask2.jpg', grains, image, scale=True)

In [ ]:
# Lineal intercept analysis (ISO 643 / ASTM E112 method)
# Uses the same pixel size as the area cell above — adjust pixel_size_nm for other images.
pixel_size_nm = 5.582
chord_lengths, mean_ic = seg.lineal_intercept_analysis(
    labels,
    angle=0,     # horizontal scan lines; change to e.g. 45 for diagonal
    n_lines=200,
    scale=pixel_size_nm,
    scale_unit='nm',
)
print(f"Mean lineal intercept l̅ = {mean_ic:.1f} nm")
print(f"ASTM grain size number G = {seg.astm_grain_size_number(mean_ic * 1e-6):.2f}")
fig_ic, ax_ic = seg.plot_histogram_of_lineal_intercepts(chord_lengths, scale_unit='nm')
plt.show()

In [ ]:
# Pixel-counting grain area analysis
# Each pixel in this SEM image is 5.582 × 5.582 nm — adjust pixel_size for other images.
pixel_size_nm = 5.582
areas, equiv_diameters = seg.pixel_area_analysis(
    labels,
    pixel_size=pixel_size_nm,
    pixel_unit='nm',
)
print(f"Grains measured: {len(areas)}")
print(f"Mean grain area:              {areas.mean():.3e} nm²")
print(f"Mean equivalent diameter:     {equiv_diameters.mean():.1f} nm")
fig_area, ax_area = seg.plot_histogram_of_grain_areas(areas, pixel_unit='nm')
plt.show()

## Delete, merge, and add grains

Open and run the [`interactive_edit.ipynb`](interactive_edit.ipynb) notebook to refine the results and generate training data.

## Fine-tune the ResNeXt

Adapt the existing checkpoint to a new image domain by continuing training on a small set of paired image / mask files. Mask files written by `seg.save_training_masks` (the cell above) drop in directly — point `input_dir` at the folder that contains them.

This mirrors `Train_ResNeXt_model.ipynb`, except training resumes from the saved weights instead of starting from the ImageNet-only encoder, and the new checkpoint is written to a separate path so the original is preserved.

In [2]:
import glob
import random

from PIL import Image
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms.functional as TF

In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Folder of paired *image* / *mask* files (e.g. output of seg.save_training_masks).
input_dir = './Images/Masks_and_images/'

# Patches written here — Patches/images and Patches/labels.
patch_dir = './Images/Resnext_Patches/'

# Starting checkpoint and where the fine-tuned model goes.
base_checkpoint = './models/resnext_model.pth'
save_path       = './models/resnext_model_finetuned.pth'

# ── Hyperparameters ───────────────────────────────────────────────────────────
val_split     = 0.15
batch_size    = 8
epochs        = 25
lr            = 5e-5         # smaller than full training to preserve learned features
patience      = 8
class_weights = (0.6, 1.0, 5.0)  # background / interior / boundary

In [4]:
# Generate 256x256 patches from the full-size images and masks.
# Skip this cell if patches already exist in patch_dir/Patches/.
image_dir, mask_dir = seg.patchify_training_data(input_dir, patch_dir)
print(f'Image patches: {image_dir}')
print(f'Mask patches:  {mask_dir}')
print(f'Patches found: {len(glob.glob(os.path.join(image_dir, "*.png")))}')

100%|██████████| 18/18 [00:00<00:00, 52.06it/s]

Image patches: ./Images/Resnext_Patches/Patches/images
Mask patches:  ./Images/Resnext_Patches/Patches/labels
Patches found: 504


In [5]:
class GrainPatchDataset(Dataset):
    """Loads paired 256x256 image/mask PNG patches produced by patchify_training_data.

    save_training_masks writes both _sam_mask and _unet_mask for each image,
    so after patchifying there are twice as many label patches as image patches.
    mask_suffix selects which set: 'sam' (cleaner outlines) or 'unet'.
    """

    def __init__(self, image_dir, mask_dir, augment=True, mask_suffix='sam'):
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, '*.png')))
        all_masks        = sorted(glob.glob(os.path.join(mask_dir,  '*.png')))

        n_total = len(all_masks)
        n_imgs  = len(self.image_paths)
        assert n_total % n_imgs == 0, (
            f'Expected label patches to be a multiple of image patches, '
            f'got {n_total} masks for {n_imgs} images.')
        factor = n_total // n_imgs
        mask_index = {'sam': 0, 'unet': 1 if factor > 1 else 0}
        offset = mask_index.get(mask_suffix, 0)
        self.mask_paths = all_masks[offset * n_imgs : (offset + 1) * n_imgs]

        assert len(self.image_paths) == len(self.mask_paths), (
            f'Mismatch: {len(self.image_paths)} images vs {len(self.mask_paths)} masks')
        self.augment = augment
        print(f'Using {mask_suffix} masks ({len(self.image_paths)} patch pairs)')

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = Image.open(self.image_paths[idx]).convert('RGB')
        mask = Image.open(self.mask_paths[idx]).convert('L')

        if self.augment:
            if random.random() > 0.5:
                img  = TF.hflip(img)
                mask = TF.hflip(mask)
            if random.random() > 0.5:
                img  = TF.vflip(img)
                mask = TF.vflip(mask)
            k = random.randint(0, 3)
            if k:
                img  = TF.rotate(img,  90 * k)
                mask = TF.rotate(mask, 90 * k)

        img_t  = torch.from_numpy(np.array(img).astype('float32') / 255.0).permute(2, 0, 1)
        mask_t = torch.from_numpy(np.array(mask).astype('int64'))
        return img_t, mask_t


full_dataset = GrainPatchDataset(image_dir, mask_dir, augment=True, mask_suffix='sam')
n_val   = max(1, int(len(full_dataset) * val_split))
n_train = len(full_dataset) - n_val
train_dataset, val_dataset = random_split(full_dataset, [n_train, n_val])
val_dataset.dataset.augment = False

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)

print(f'Training patches:   {n_train}')
print(f'Validation patches: {n_val}')

Using sam masks (504 patch pairs)
Training patches:   429
Validation patches: 75


In [6]:
# Load the existing checkpoint as the starting point for fine-tuning.
ft_model = MaskingResNeXt(num_classes=3, pretrained=False)
ft_model.load_state_dict(torch.load(base_checkpoint, map_location=device))
ft_model = ft_model.to(device)

optimizer = torch.optim.Adam(ft_model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=patience // 2, factor=0.5
)

total_params = sum(p.numel() for p in ft_model.parameters())
trainable    = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

Total parameters:     94,251,683
Trainable parameters: 94,251,683


In [7]:
train_losses, val_losses = [], []
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(epochs):
    # ── training ──────────────────────────────────────────────────────────────
    ft_model.train()
    running_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [train]', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = ft_model(imgs)
        loss  = weighted_crossentropy_torch(preds, masks, class_weights, device)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / n_train

    # ── validation ────────────────────────────────────────────────────────────
    ft_model.eval()
    running_val = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = ft_model(imgs)
            loss  = weighted_crossentropy_torch(preds, masks, class_weights, device)
            running_val += loss.item() * imgs.size(0)
    val_loss = running_val / n_val

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    print(f'Epoch {epoch+1:3d}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        torch.save(ft_model.state_dict(), save_path)
        print(f'  saved best model (val_loss={best_val_loss:.4f})')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping after {epoch+1} epochs (no improvement for {patience} epochs).')
            break

print(f'\nBest validation loss: {best_val_loss:.4f}')
print(f'Model saved to: {save_path}')

Epoch   1/25  train_loss=0.9322  val_loss=0.7733
  saved best model (val_loss=0.7733)


Epoch   2/25  train_loss=0.7619  val_loss=0.7092
  saved best model (val_loss=0.7092)


Epoch   3/25  train_loss=0.7501  val_loss=0.7025
  saved best model (val_loss=0.7025)


Epoch   4/25  train_loss=0.7334  val_loss=0.6981
  saved best model (val_loss=0.6981)


Epoch   5/25  train_loss=0.7104  val_loss=0.6816
  saved best model (val_loss=0.6816)


...

Epoch  21/25  train_loss=0.6016  val_loss=0.6161
  saved best model (val_loss=0.6161)


Epoch  22/25  train_loss=0.5966  val_loss=0.6196


Epoch  23/25  train_loss=0.5968  val_loss=0.6134
  saved best model (val_loss=0.6134)


Epoch  24/25  train_loss=0.5934  val_loss=0.6101
  saved best model (val_loss=0.6101)


Epoch  25/25  train_loss=0.5953  val_loss=0.6076
  saved best model (val_loss=0.6076)

Best validation loss: 0.6076
Model saved to: ./models/resnext_model_finetuned.pth


In [9]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train loss')
ax.plot(val_losses,   label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Weighted cross-entropy loss')
ax.set_title('ResNeXt fine-tuning')
ax.legend()
plt.tight_layout()
plt.show()

### Use the fine-tuned model

Reload the fine-tuned checkpoint and re-wrap it in the adapter, then re-run the segmentation cells above.

In [10]:
resnext = load_resnext(save_path, device=device).to(device)
resnext.eval()
resnext_adapter = TorchKerasAdapter(resnext, device)

NameError: name 'TorchKerasAdapter' is not defined